# Gemma3-SD: Dual CLIP Scaffold → Native Gemma Conditioning for Stable Diffusion 1.5

Goal: migrate SD1.5 from CLIP conditioning to Gemma 3 270M native conditioning without relying on a CLIP-space linear bake.

Transition path:

```text
SD1.5 CLIP path intact
    + native Gemma cross-attention branch
    → train Gemma branch under CLIP scaffold
    → decay CLIP-teacher loss while student stays Gemma-only
    → prune to Gemma-only inference
```

CLIP is a training scaffold and teacher. Final target is no CLIP at inference.


## Section 0: Google Drive Mount

Mount Google Drive for saving all artifacts (probe, LoRA, samples, complete model).

In [ ]:
# @title 0.1 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUT = '/content/drive/MyDrive/gemma3-sd'
os.makedirs(DRIVE_OUT, exist_ok=True)
print(f"Artifacts will be saved to: {DRIVE_OUT}")


## Section 1: Environment Setup

In [ ]:
# @title 1.1 Simple install — Colab + Kohya, no Torch/NumPy changes

!pip install -q -U --upgrade-strategy only-if-needed \
  "Pillow==11.3.0" \
  "accelerate==1.6.0" \
  "transformers==4.54.1" \
  "diffusers[torch]==0.32.1" \
  "safetensors==0.4.5" \
  "datasets" \
  "peft" \
  "bitsandbytes" \
  "ftfy" \
  "einops" \
  "opencv-python==4.10.0.84" \
  "lion-pytorch" \
  "schedulefree" \
  "pytorch-optimizer" \
  "prodigyopt" \
  "prodigy-plus-schedule-free" \
  "toml" \
  "voluptuous" \
  "imagesize" \
  "rich" \
  "sentencepiece" \
  "wandb" \
  "matplotlib" \
  "tensorboard" \
  "tqdm"

!git clone --depth 1 https://github.com/kohya-ss/sd-scripts.git /content/sd-scripts 2>/dev/null || true
!pip install -q --no-deps -e /content/sd-scripts

In [ ]:
import torch, numpy as np, PIL
import transformers, diffusers, accelerate

print("OK")
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("NumPy:", np.__version__)
print("Pillow:", PIL.__version__)
print("Transformers:", transformers.__version__)
print("Diffusers:", diffusers.__version__)

In [ ]:
# @title 1.3 HuggingFace Login via Colab Secrets
from google.colab import userdata
import os
hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("Missing HF_TOKEN in Colab secrets. Add it via the key icon in the sidebar.")
os.environ["HF_TOKEN"] = hf_token
print("HF_TOKEN: verified")
print("HF_TOKEN loaded from Colab secrets.")
print("Make sure you accepted the Gemma license: https://huggingface.co/google/gemma-3-270m-it")


In [ ]:
# @title 1.4 Run Configuration + wandb via Colab Secrets
import wandb
import os
from datetime import datetime

# Modes:
#   diagnostic  = run smoke diagnostics only; training cells skip.
#   short_train = diagnostics + modest train run.
#   full_train  = diagnostics + longer train run using same code paths.
RUN_MODE = "diagnostic"  # @param ["diagnostic", "short_train", "full_train"]
RUN_TRAINING = RUN_MODE in {"short_train", "full_train"}
RUN_DIAGNOSTICS = True
RUN_FINAL_PROOF = True
RUN_FIXED_VALIDATION_GRIDS = True

BASE_SEED = 1234
MAX_SAMPLES_WARMUP = 6_000 if RUN_MODE != "diagnostic" else 128
MAX_SAMPLES_PHASEB = 6_000 if RUN_MODE != "diagnostic" else 128
FULLRANK_EPOCHS = 2 if RUN_MODE == "full_train" else (1 if RUN_MODE == "short_train" else 0)
PHASEB_EPOCHS = 1 if RUN_MODE in {"short_train", "full_train"} else 0

# Optimizer-step limits. short_train is intentionally a controlled falsifier run,
# not a 6k+6k blind training run. DataLoader max_samples remains a ceiling; these
# step limits stop earlier and drive validation-grid cadence.
PHASE_A_MAX_OPT_STEPS = 100 if RUN_MODE == "short_train" else (None if RUN_MODE == "full_train" else 0)
PHASE_B_MAX_OPT_STEPS = 200 if RUN_MODE == "short_train" else (None if RUN_MODE == "full_train" else 0)
TEACHER_DECAY_STEPS = 1000 if RUN_MODE == "full_train" else 200
VALIDATION_EVERY_OPT_STEPS = 50 if RUN_MODE == "short_train" else (250 if RUN_MODE == "full_train" else 0)
VALIDATION_TRAINING_STEPS = 20  # cheaper than final 30-step proof grids

FULLRANK_LR = 1e-5
PHASEB_LR = 5e-5
TRAIN_BATCH_SIZE = 2  # dataset batch; text-delta internally runs cond/uncond pairs, so UNet batch is 2x this
GRADIENT_ACCUMULATION_STEPS = 4
LAMBDA_DIFFUSION = 0.25
LAMBDA_TEXT_DELTA = 1.0
TEXT_DELTA_ENABLED = True

SHUFFLE_STREAMING = True
SHUFFLE_BUFFER = 10_000
SKIP_CLIP_COMPUTE_WHEN_ZERO = True

VAL_PROMPTS = [
    "a cat sitting on a windowsill looking outside",
    "a watercolor painting of a mountain lake",
    "a neon-lit cyberpunk alleyway at night",
]
DIAGNOSTIC_PROMPTS = [
    "a red sports car on a city street",
    "a snowy mountain at dawn",
    "an underwater coral reef with colorful fish",
    "a rustic Italian kitchen with tomatoes",
    "abstract geometric shapes in neon colors",
]
VAL_SEED = 42
VAL_STEPS = 30
VAL_GUIDANCE = 5.0

RUN_CONFIG = {
    "run_mode": RUN_MODE,
    "model": "Gemma 3 270M → SD 1.5 dual CLIP-teacher/Gemma-student",
    "gemma_hidden_size": 640,
    "original_cross_attn_dim": 768,
    "new_cross_attn_dim": 640,
    "max_prompt_length": 77,
    "fullrank_epochs": FULLRANK_EPOCHS,
    "phaseb_epochs": PHASEB_EPOCHS,
    "max_samples_warmup": MAX_SAMPLES_WARMUP,
    "max_samples_phaseb": MAX_SAMPLES_PHASEB,
    "fullrank_lr": FULLRANK_LR,
    "phaseb_lr": PHASEB_LR,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "paired_unet_batch_size": 2 * TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "phase_a_max_opt_steps": PHASE_A_MAX_OPT_STEPS,
    "phase_b_max_opt_steps": PHASE_B_MAX_OPT_STEPS,
    "teacher_decay_steps": TEACHER_DECAY_STEPS,
    "validation_every_opt_steps": VALIDATION_EVERY_OPT_STEPS,
    "validation_training_steps": VALIDATION_TRAINING_STEPS,
    "lambda_diffusion": LAMBDA_DIFFUSION,
    "lambda_text_delta": LAMBDA_TEXT_DELTA,
    "text_delta_enabled": TEXT_DELTA_ENABLED,
    "shuffle_streaming": SHUFFLE_STREAMING,
    "shuffle_buffer": SHUFFLE_BUFFER,
    "skip_clip_compute_when_zero": SKIP_CLIP_COMPUTE_WHEN_ZERO,
    "phase_a_trainables": "gemma_norm + gemma_attn.to_k/to_v full-rank",
    "phase_b_trainables": "K/V LoRA + gemma_norm + gemma_attn.to_out[0]",
    "device": str(torch.cuda.get_device_name(0)) if torch.cuda.is_available() else "cpu",
}

# Load wandb key from Colab secrets
wb_key = userdata.get("WANDB_API_KEY") or userdata.get("WANDB_KEY")
if wb_key is None:
    raise ValueError("Missing WANDB_API_KEY or WANDB_KEY in Colab secrets.")
os.environ["WANDB_API_KEY"] = wb_key
print("WANDB_API_KEY loaded from Colab secrets.")

run_name = f"gemma3-sd-{RUN_MODE}-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
wandb.init(project="gemma3-stable-diffusion", name=run_name, config=RUN_CONFIG)
print(f"wandb run: {wandb.run.name}")
print("Run config:")
for k, v in RUN_CONFIG.items():
    print(f"  {k}: {v}")


In [ ]:
# @title 2.1 Load Gemma 3 270M
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda")
MAX_GEMMA_LEN = 77  # Start at SD1.5's CLIP length. Extend only after Gemma-only works.
GEMMA_LAYER_INDEX = -1  # TODO experiment with middle/upper layers after baseline works.

print("Loading Gemma 3 270M...")
gemma_path = "google/gemma-3-270m"
gemma_tokenizer = AutoTokenizer.from_pretrained(gemma_path, token=os.environ.get("HF_TOKEN"))
if gemma_tokenizer.pad_token is None:
    gemma_tokenizer.pad_token = gemma_tokenizer.eos_token

gemma_model = AutoModelForCausalLM.from_pretrained(
    gemma_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=os.environ.get("HF_TOKEN"),
    low_cpu_mem_usage=True,
).eval()
for p in gemma_model.parameters():
    p.requires_grad = False

gemma_hidden_size = gemma_model.config.hidden_size  # 640 for Gemma 3 270M
print(f"  Gemma hidden_size: {gemma_hidden_size}")
print(f"  Gemma dtype: {next(gemma_model.parameters()).dtype}")
print(f"  Gemma layer index for conditioning: {GEMMA_LAYER_INDEX}")
print("  Gemma frozen. Native Gemma cross-attention branch will learn to read these states.")


## Section 2B: Dual Encoder Setup

CLIP remains loaded during transition as a scaffold/teacher. Gemma gets its own native cross-attention branch; we do not solve a static CLIP/Gemma linear mapping.


In [ ]:
# @title 2.2 Load CLIP Persistently (training scaffold / teacher)
from transformers import CLIPTextModel, CLIPTokenizer

CLIP_ID = "openai/clip-vit-large-patch14"
print(f"Loading CLIP scaffold: {CLIP_ID}")
clip_tokenizer = CLIPTokenizer.from_pretrained(CLIP_ID)
clip_model = CLIPTextModel.from_pretrained(
    CLIP_ID,
    torch_dtype=torch.float16,
).to(device).eval()
for p in clip_model.parameters():
    p.requires_grad = False

clip_hidden_size = clip_model.config.hidden_size  # 768 for ViT-L/14
print(f"  CLIP hidden size: {clip_hidden_size}")
print(f"  Gemma hidden size: {gemma_hidden_size}")
assert clip_hidden_size == 768, "SD1.5 UNet expects CLIP hidden size 768"
assert gemma_hidden_size == 640, "Gemma 3 270M expected hidden size 640"
print("  CLIP frozen. It is a teacher/scaffold only; final inference uses the pruned Gemma-only checkpoint.")


## Legacy linear bake removed

The previous static ridge/bake route is intentionally removed from the mainline. Runtime evidence showed the linear map collapsed to near-zero signal, so this notebook now learns native Gemma attention behavior through the UNet instead.


## No static embedding target

We do not train Gemma to imitate CLIP embeddings. Gemma remains frozen and supplies hidden states; the UNet learns a native Gemma-conditioning branch under a frozen CLIP teacher/scaffold.


## Section 3: Dual Cross-Attention UNet Setup

Every SD1.5 cross-attention block keeps its original CLIP `attn2` path and gains a parallel Gemma-native attention path. `clip_scale` and `gemma_scale` control the transition.


In [ ]:
# @title 3.1 Load StyleJourney SD UNet + VAE + Scheduler from Colab safetensors
from diffusers import StableDiffusionPipeline, UNet2DConditionModel, AutoencoderKL, DDPMScheduler
import os, gc

# Local Colab checkpoint requested for this run.
# Put the file at /content/models/stylejourney_v10.safetensors.
# If your Colab runtime mounted it as /models/stylejourney_v10.safetensors, the resolver below also accepts that.
SD_BASE_ID = "runwayml/stable-diffusion-v1-5"  # only a compatibility label/fallback for SD1.5-shaped config
SD_CHECKPOINT_CANDIDATES = [
    "/content/drive/MyDrive/model/stylejourney_v10.safetensors",
    "/models/stylejourney_v10.safetensors",
]


def resolve_sd_checkpoint():
    for path in SD_CHECKPOINT_CANDIDATES:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        "stylejourney_v10.safetensors not found. Upload/copy it to one of: "
        + ", ".join(SD_CHECKPOINT_CANDIDATES)
    )


SD_CHECKPOINT = resolve_sd_checkpoint()
SD_ID = SD_CHECKPOINT  # downstream metadata/reload cells should point at the actual local checkpoint
print(f"Loading Stable Diffusion checkpoint from: {SD_CHECKPOINT}")


def load_stylejourney_components(torch_dtype=torch.float32, target_device=device):
    """Load local StyleJourney safetensors once, then extract UNet/VAE/scheduler.

    Diffusers loads .safetensors checkpoints through StableDiffusionPipeline.from_single_file.
    We immediately drop text encoder/tokenizer/safety-checker because this notebook uses
    its own CLIP teacher and final Gemma-only conditioning path.
    """
    pipe = StableDiffusionPipeline.from_single_file(
        SD_CHECKPOINT,
        torch_dtype=torch_dtype,
        safety_checker=None,
        requires_safety_checker=False,
    )
    base_unet = pipe.unet.to(target_device)
    base_vae = pipe.vae.to(target_device).eval()
    train_scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

    # Drop unused pipeline parts to reduce VRAM/RAM pressure after extracting modules.
    pipe.text_encoder = None
    pipe.tokenizer = None
    pipe.safety_checker = None
    del pipe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return base_unet, base_vae, train_scheduler


# Keep UNet/VAE fp32 for stability; trainable Gemma branch is small enough for L4/T4 class GPUs.
unet, vae, scheduler = load_stylejourney_components(torch_dtype=torch.float32, target_device=device)

unet_dtype = next(unet.parameters()).dtype
print(f"  UNet cross_attention_dim: {unet.config.cross_attention_dim}")  # 768 for SD1.x checkpoints
print(f"  UNet dtype: {unet_dtype}")
print(f"  VAE dtype: {next(vae.parameters()).dtype}")
print("  Original CLIP-sized cross-attention remains intact, initialized from StyleJourney.")


In [ ]:
# @title 3.2 UNet Surgery: Add Dual Native Gemma Cross-Attention
import math
import torch.nn as nn
from diffusers.models.attention_processor import Attention

class DualNativeAttention(nn.Module):
    """Original CLIP cross-attn plus parallel native Gemma cross-attn.

    CLIP path is the untouched SD1.5 attn2 module.
    Gemma path is a new Attention module with cross_attention_dim=gemma_hidden_size.
    The output is a scheduled weighted sum. CLIP can later be pruned structurally.
    """
    def __init__(self, clip_attn, gemma_dim, dtype=None, device=None):
        super().__init__()
        self.clip_attn = clip_attn
        self.gemma_dim = int(gemma_dim)
        query_dim = clip_attn.to_q.in_features
        inner_dim = clip_attn.to_k.out_features
        heads = int(clip_attn.heads)
        dim_head = inner_dim // heads
        out_bias = clip_attn.to_out[0].bias is not None
        dtype = dtype or clip_attn.to_k.weight.dtype
        device = device or clip_attn.to_k.weight.device

        self.gemma_norm = nn.LayerNorm(self.gemma_dim, device=device, dtype=dtype)
        self.gemma_attn = Attention(
            query_dim=query_dim,
            cross_attention_dim=self.gemma_dim,
            heads=heads,
            dim_head=dim_head,
            bias=True,       # Keep Gemma K/V bias; recreate Q below if source Q is biasless.
            out_bias=out_bias,
        ).to(device=device, dtype=dtype)

        def clone_linear_shape(src_linear):
            return nn.Linear(
                src_linear.in_features,
                src_linear.out_features,
                bias=src_linear.bias is not None,
                device=device,
                dtype=dtype,
            )

        # Copy SD1.5 image-side projections (Q and OUT) into Gemma branch.
        # Preserve source bias/no-bias exactly; Diffusers SD1.5 Q is often biasless,
        # while Attention(bias=True) would otherwise create a mismatched Q bias.
        self.gemma_attn.to_q = clone_linear_shape(self.clip_attn.to_q)
        self.gemma_attn.to_q.load_state_dict(self.clip_attn.to_q.state_dict(), strict=True)
        self.gemma_attn.to_out[0].load_state_dict(self.clip_attn.to_out[0].state_dict(), strict=True)
        for mod in [self.gemma_attn.to_k, self.gemma_attn.to_v]:
            if hasattr(mod, "weight") and mod.weight is not None:
                nn.init.normal_(mod.weight, mean=0.0, std=0.02)
            if hasattr(mod, "bias") and mod.bias is not None:
                nn.init.zeros_(mod.bias)

        self.register_buffer("clip_scale", torch.tensor(1.0, dtype=torch.float32), persistent=True)
        self.register_buffer("gemma_scale", torch.tensor(0.0, dtype=torch.float32), persistent=True)

    def set_scales(self, clip_scale=None, gemma_scale=None):
        if clip_scale is not None:
            self.clip_scale.fill_(float(clip_scale))
        if gemma_scale is not None:
            self.gemma_scale.fill_(float(gemma_scale))

    def forward(self, hidden_states, encoder_hidden_states=None, attention_mask=None, **cross_attention_kwargs):
        # Do not mutate the shared kwargs dict flowing through all transformer blocks.
        kwargs = dict(cross_attention_kwargs or {})
        # Prefer module-local context to avoid global cross_attention_kwargs leaking into
        # Diffusers self-attention processors and causing repeated "ignored" warnings.
        gemma_encoder_hidden_states = kwargs.pop("gemma_encoder_hidden_states", None)
        if gemma_encoder_hidden_states is None:
            gemma_encoder_hidden_states = getattr(self, "_gemma_encoder_hidden_states", None)
        gemma_attention_mask = kwargs.pop("gemma_attention_mask", None)
        if gemma_attention_mask is None:
            gemma_attention_mask = getattr(self, "_gemma_attention_mask", None)
        clip_scale = float(kwargs.pop("clip_scale", self.clip_scale.item()))
        gemma_scale = float(kwargs.pop("gemma_scale", self.gemma_scale.item()))

        # When clip_scale == 0 the student path is genuinely CLIP-free inside
        # the attention block. The caller may still pass CLIP-shaped tensors for
        # Diffusers API compatibility, but this branch does not compute CLIP attn.
        skip_clip_compute = globals().get("SKIP_CLIP_COMPUTE_WHEN_ZERO", True)
        clip_out = None
        if clip_scale != 0.0 or not skip_clip_compute:
            clip_out = self.clip_attn(
                hidden_states,
                encoder_hidden_states=encoder_hidden_states,
                attention_mask=attention_mask,
                **kwargs,
            )

        if gemma_encoder_hidden_states is None or gemma_scale == 0.0:
            if clip_out is None:
                raise ValueError("No active conditioning path: clip_scale=0 and gemma path disabled")
            return clip_out * clip_scale

        gemma_states = self.gemma_norm(gemma_encoder_hidden_states.to(dtype=hidden_states.dtype))
        if gemma_attention_mask is not None:
            # Side-channel Gemma masks bypass UNet's encoder_attention_mask preprocessing.
            # Tokenizers return int64 masks; Torch SDPA requires bool/float/query dtype.
            # Boolean keeps tokenizer semantics: True/1 = attend, False/0 = masked.
            gemma_attention_mask = gemma_attention_mask.to(device=hidden_states.device, dtype=torch.bool)
        gemma_out = self.gemma_attn(
            hidden_states,
            encoder_hidden_states=gemma_states,
            attention_mask=gemma_attention_mask,
            **kwargs,
        )
        if clip_out is None:
            return gemma_out * gemma_scale
        return clip_out * clip_scale + gemma_out * gemma_scale


class GemmaOnlyAttention(nn.Module):
    """Pruned inference wrapper: preserves trained Gemma normalization plus Gemma attention."""
    def __init__(self, gemma_norm, gemma_attn):
        super().__init__()
        self.gemma_norm = gemma_norm
        self.gemma_attn = gemma_attn

    def forward(self, hidden_states, encoder_hidden_states=None, attention_mask=None, **cross_attention_kwargs):
        if encoder_hidden_states is None:
            raise ValueError("GemmaOnlyAttention requires Gemma encoder_hidden_states")
        gemma_states = self.gemma_norm(encoder_hidden_states.to(dtype=hidden_states.dtype))
        if attention_mask is not None and attention_mask.dtype not in (torch.bool, hidden_states.dtype):
            # Pruned Gemma-only paths may receive tokenizer int64 masks directly.
            attention_mask = attention_mask.to(device=hidden_states.device, dtype=torch.bool)
        elif attention_mask is not None:
            attention_mask = attention_mask.to(device=hidden_states.device)
        kwargs = dict(cross_attention_kwargs or {})
        kwargs.pop("gemma_encoder_hidden_states", None)
        kwargs.pop("gemma_attention_mask", None)
        kwargs.pop("clip_scale", None)
        kwargs.pop("gemma_scale", None)
        return self.gemma_attn(
            hidden_states,
            encoder_hidden_states=gemma_states,
            attention_mask=attention_mask,
            **kwargs,
        )


def is_dual_native_attention_module(module):
    """Robust across notebook cell re-execution where class identity changes."""
    return (
        hasattr(module, "clip_attn")
        and hasattr(module, "gemma_attn")
        and hasattr(module, "gemma_norm")
        and hasattr(module, "set_scales")
    )


def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def print_trainable_summary(model, label="trainable summary"):
    groups = {
        "gemma_norm": 0,
        "gemma_to_k": 0,
        "gemma_to_v": 0,
        "gemma_to_q": 0,
        "gemma_to_out": 0,
        "lora": 0,
        "other_trainable": 0,
    }
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        n = p.numel()
        if "lora_" in name:
            groups["lora"] += n
        elif "gemma_norm" in name:
            groups["gemma_norm"] += n
        elif "gemma_attn.to_k" in name:
            groups["gemma_to_k"] += n
        elif "gemma_attn.to_v" in name:
            groups["gemma_to_v"] += n
        elif "gemma_attn.to_q" in name:
            groups["gemma_to_q"] += n
        elif "gemma_attn.to_out" in name:
            groups["gemma_to_out"] += n
        else:
            groups["other_trainable"] += n
    trainable, total = count_trainable_params(model)
    print(f"{label}: {trainable:,} / {total:,} trainable ({100*trainable/max(total,1):.4f}%)")
    for k, v in groups.items():
        if v:
            print(f"  {k}: {v:,}")
    wandb.log({f"params/{label.replace(' ', '_')}_trainable": trainable,
               f"params/{label.replace(' ', '_')}_total": total,
               **{f"params/{label.replace(' ', '_')}_{k}": v for k, v in groups.items()}})
    return groups


def count_modules_by_predicate(model, pred):
    return sum(1 for module in model.modules() if pred(module))


def is_gemma_only_attention_module(module):
    """Structural predicate robust to Colab class re-execution."""
    return (
        hasattr(module, "gemma_attn")
        and hasattr(module, "gemma_norm")
        and not hasattr(module, "clip_attn")
    )


def apply_dual_native_attention(model, gemma_dim):
    replacements = 0
    for name, module in model.named_modules():
        if hasattr(module, "attn2") and module.attn2 is not None:
            if not is_dual_native_attention_module(module.attn2):
                module.attn2 = DualNativeAttention(
                    module.attn2,
                    gemma_dim=gemma_dim,
                    dtype=next(model.parameters()).dtype,
                    device=next(model.parameters()).device,
                )
                replacements += 1
    return replacements


def set_dual_attention_scales(model, clip_scale=1.0, gemma_scale=0.0):
    count = 0
    for module in model.modules():
        if is_dual_native_attention_module(module):
            module.set_scales(clip_scale=clip_scale, gemma_scale=gemma_scale)
            count += 1
    return count


def set_dual_attention_context(model, gemma_encoder_hidden_states=None, gemma_attention_mask=None, clip_scale=None, gemma_scale=None):
    """Set Gemma context directly on dual-attn modules; avoids noisy global cross_attention_kwargs."""
    count = 0
    for module in model.modules():
        if is_dual_native_attention_module(module):
            module._gemma_encoder_hidden_states = gemma_encoder_hidden_states
            module._gemma_attention_mask = gemma_attention_mask
            module.set_scales(clip_scale=clip_scale, gemma_scale=gemma_scale)
            count += 1
    if count == 0:
        raise RuntimeError("No dual native attention modules found while setting context")
    return count


def clear_dual_attention_context(model):
    for module in model.modules():
        if is_dual_native_attention_module(module):
            module._gemma_encoder_hidden_states = None
            module._gemma_attention_mask = None


def freeze_all_but_gemma_branch(model):
    """Phase A trainables: gemma_norm + gemma_attn.to_k/to_v only (freeze to_q/to_out initially)."""
    for p in model.parameters():
        p.requires_grad = False
    trainable = []
    wrapped = 0
    for name, module in model.named_modules():
        if is_dual_native_attention_module(module):
            wrapped += 1
            for p in module.gemma_norm.parameters():
                p.requires_grad = True
                trainable.append(p)
            for p in module.gemma_attn.to_k.parameters():
                p.requires_grad = True
                trainable.append(p)
            for p in module.gemma_attn.to_v.parameters():
                p.requires_grad = True
                trainable.append(p)
    if wrapped == 0:
        raise RuntimeError("No dual native attention modules found. Rerun cells 3.1, 3.2, 3.3 before training; do not run save/prune before warmup.")
    if len(trainable) == 0:
        raise RuntimeError(f"Found {wrapped} dual modules but zero trainable Gemma params")
    return trainable


def unfreeze_gemma_to_out(model):
    """Phase B optional: additionally unfreeze Gemma branch output projection."""
    trainable = []
    wrapped = 0
    for module in model.modules():
        if is_dual_native_attention_module(module):
            wrapped += 1
            for p in module.gemma_attn.to_out[0].parameters():
                p.requires_grad = True
                trainable.append(p)
    if wrapped == 0:
        raise RuntimeError("No dual native attention modules found")
    return trainable


def prune_to_gemma_only(model, gemma_dim=None):
    """Replace DualNativeAttention wrappers with GemmaOnlyAttention for final no-CLIP inference."""
    gemma_dim = int(gemma_dim or gemma_hidden_size)
    count = 0
    for name, module in model.named_modules():
        if hasattr(module, "attn2") and is_dual_native_attention_module(module.attn2):
            module.attn2 = GemmaOnlyAttention(module.attn2.gemma_norm, module.attn2.gemma_attn)
            count += 1
    model.register_to_config(cross_attention_dim=gemma_dim, pruned_gemma_only=True)
    return count

num_dual = apply_dual_native_attention(unet, gemma_hidden_size)
set_dual_attention_scales(unet, clip_scale=1.0, gemma_scale=0.0)
unet.register_to_config(dual_native_attention=True, gemma_cross_attention_dim=gemma_hidden_size, uses_kv_bias=True)

print(f"  DualNativeAttention wrappers installed: {num_dual}")
print("  Expected SD1.5 cross-attn module count depends on Diffusers/model config; live count is authoritative.")
assert num_dual > 0, "No cross-attention modules were wrapped"
# SD1.5 variants/runtime configs commonly report 13 or 16 attn2 modules. Do not hardcode.
unet.register_to_config(dual_native_attn_count=num_dual)
for n, p in unet.named_parameters():
    assert torch.isfinite(p).all(), f"{n} has NaN/Inf"
print("  All parameters finite. CLIP path preserved, Gemma branch initially scaled to zero.")


In [ ]:
# @title 3.3 Verify Dual Forward + Mandatory Pre-Training Diagnostics
import torch.nn.functional as F


def _as_prompt_list(prompts):
    if isinstance(prompts, str):
        return [prompts]
    return list(prompts)

@torch.no_grad()
def encode_clip_prompts(prompts):
    prompts = _as_prompt_list(prompts)
    tok = clip_tokenizer(
        prompts,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=clip_tokenizer.model_max_length,
    ).to(device)
    hidden = clip_model(**tok).last_hidden_state.to(device=device, dtype=unet_dtype)
    assert torch.isfinite(hidden).all(), "CLIP hidden states have NaN/Inf"
    return hidden, tok.attention_mask

@torch.no_grad()
def encode_gemma_prompts(prompts, layer_index=None, return_all_layers=False):
    prompts = _as_prompt_list(prompts)
    tok = gemma_tokenizer(
        prompts,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=MAX_GEMMA_LEN,
    ).to(device)
    out = gemma_model(input_ids=tok.input_ids, attention_mask=tok.attention_mask, output_hidden_states=True)
    if return_all_layers:
        hidden_states = [h.to(device=device, dtype=unet_dtype) for h in out.hidden_states]
        for h in hidden_states:
            assert torch.isfinite(h).all(), "Gemma hidden states have NaN/Inf"
        return hidden_states, tok.attention_mask
    idx = GEMMA_LAYER_INDEX if layer_index is None else layer_index
    hidden = out.hidden_states[idx].to(device=device, dtype=unet_dtype)
    assert torch.isfinite(hidden).all(), "Gemma hidden states have NaN/Inf"
    return hidden, tok.attention_mask

@torch.no_grad()
def compute_dual_prompt_sensitivity(prompts, seed=123, timestep=500, clip_scale=0.0, gemma_scale=1.0, label="sensitivity"):
    gen = torch.Generator(device=device).manual_seed(seed)
    latent = torch.randn(1, 4, 64, 64, generator=gen, device=device, dtype=unet_dtype)
    t = torch.tensor([timestep], device=device)
    preds = []
    for ptxt in prompts:
        clip_h, clip_mask = encode_clip_prompts([ptxt])
        gemma_h, gemma_mask = encode_gemma_prompts([ptxt])
        set_dual_attention_context(unet, gemma_encoder_hidden_states=gemma_h, gemma_attention_mask=gemma_mask, clip_scale=clip_scale, gemma_scale=gemma_scale)
        pred = unet(latent, t, encoder_hidden_states=clip_h, encoder_attention_mask=clip_mask).sample.float()
        preds.append(pred)
    base = preds[0].pow(2).mean().sqrt().item() + 1e-8
    vals = []
    for i in range(len(prompts)):
        for j in range(i + 1, len(prompts)):
            diff = (preds[i] - preds[j]).pow(2).mean().sqrt().item() / base
            vals.append(diff)
            print(f"[{label}] {i} vs {j}: relative diff = {diff:.6f}")
    mean_val = float(np.mean(vals)) if vals else 0.0
    wandb.log({f"diagnostics/{label}_mean_relative_diff": mean_val})
    return mean_val

@torch.no_grad()
def gemma_layer_discriminability(prompts, layer_indices=(4, 8, 12, 16, 20, -1)):
    prompts = _as_prompt_list(prompts)
    all_layers, mask = encode_gemma_prompts(prompts, return_all_layers=True)
    n_layers = len(all_layers)
    print(f"Gemma hidden_states available: {n_layers} entries; requested sweep={list(layer_indices)}")

    # Model variants expose different hidden_states lengths. Resolve negatives and
    # skip out-of-range diagnostic indices instead of crashing the notebook.
    valid_indices = []
    for raw_idx in layer_indices:
        resolved = raw_idx if raw_idx >= 0 else n_layers + raw_idx
        if 0 <= resolved < n_layers:
            if resolved not in valid_indices:
                valid_indices.append(resolved)
        else:
            print(f"Skipping Gemma layer {raw_idx}: out of range for {n_layers} hidden_states entries")
    if not valid_indices:
        valid_indices = [n_layers - 1]
        print(f"No requested layers were valid; falling back to final hidden_state index {valid_indices[0]}")

    mask_f = mask.to(device=device, dtype=torch.float32).unsqueeze(-1)
    results = {}
    for idx in valid_indices:
        h = all_layers[idx].float()
        h = F.layer_norm(h, (h.shape[-1],))
        pooled = (h * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp_min(1.0)
        pooled = F.normalize(pooled, dim=-1)
        sims = pooled @ pooled.T
        tri = torch.triu_indices(len(prompts), len(prompts), offset=1, device=sims.device)
        mean_cos = sims[tri[0], tri[1]].mean().item() if tri.numel() else 1.0
        token_var = h.var(dim=1).mean().item()
        results[int(idx)] = {"mean_pairwise_cos": mean_cos, "token_var": token_var}
        print(f"Gemma layer {idx:>3}: mean pairwise cos={mean_cos:.4f}, token_var={token_var:.4f}")
        wandb.log({f"diagnostics/gemma_layer_{idx}_mean_cos": mean_cos,
                   f"diagnostics/gemma_layer_{idx}_token_var": token_var})
    return results

@torch.no_grad()
def gemma_attention_score_diagnostic(prompt=None, timestep=500):
    """Approximate Gemma cross-attention entropy from Q·K scores for the first dual module.

    Diffusers SDPA processors do not expose attention weights, so this diagnostic
    computes score entropy directly from the copied Gemma to_q and trainable to_k.
    It is a mask/scale sanity check, not a replacement for full processor tracing.
    """
    prompt = prompt or DIAGNOSTIC_PROMPTS[0]
    first_dual = next((m for m in unet.modules() if is_dual_native_attention_module(m)), None)
    if first_dual is None:
        raise RuntimeError("No DualNativeAttention module for attention diagnostic")
    gh, gm = encode_gemma_prompts([prompt])
    gh_norm = first_dual.gemma_norm(gh.to(dtype=unet_dtype))
    latent = torch.randn(1, 4, 64, 64, device=device, dtype=unet_dtype)
    t = torch.tensor([timestep], device=device)
    ch, cm = encode_clip_prompts([prompt])
    set_dual_attention_context(unet, gemma_encoder_hidden_states=gh, gemma_attention_mask=gm, clip_scale=0.0, gemma_scale=1.0)
    # Get a representative hidden_state shape by running to the first block is invasive;
    # instead use random hidden states matching this module's query_dim for score-scale sanity.
    query_dim = first_dual.gemma_attn.to_q.in_features
    n_query = 64
    hs = torch.randn(1, n_query, query_dim, device=device, dtype=unet_dtype)
    q = first_dual.gemma_attn.to_q(hs).float()
    k = first_dual.gemma_attn.to_k(gh_norm).float()
    heads = int(first_dual.gemma_attn.heads)
    head_dim = q.shape[-1] // heads
    q = q.view(1, n_query, heads, head_dim).transpose(1, 2)
    k = k.view(1, k.shape[1], heads, head_dim).transpose(1, 2)
    scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(head_dim)
    attend = gm.to(device=device, dtype=torch.bool).view(1, 1, 1, -1)
    scores = scores.masked_fill(~attend, -1e4)
    probs = scores.softmax(dim=-1)
    entropy = -(probs.clamp_min(1e-8) * probs.clamp_min(1e-8).log()).sum(dim=-1).mean().item()
    active_tokens = int(gm.sum().item())
    max_entropy = math.log(max(active_tokens, 1))
    ratio = entropy / max(max_entropy, 1e-8)
    print(f"Gemma attention score entropy: {entropy:.4f} / log(active_tokens)={max_entropy:.4f} ratio={ratio:.4f}; active_tokens={active_tokens}")
    wandb.log({"diagnostics/gemma_score_entropy": entropy,
               "diagnostics/gemma_score_entropy_ratio": ratio,
               "diagnostics/gemma_active_tokens": active_tokens})
    return {"entropy": entropy, "entropy_ratio": ratio, "active_tokens": active_tokens}

@torch.no_grad()
def teacher_text_delta_baseline(prompt=None, seed=123, timestep=500):
    prompt = prompt or DIAGNOSTIC_PROMPTS[0]
    gen = torch.Generator(device=device).manual_seed(seed)
    latent = torch.randn(1, 4, 64, 64, generator=gen, device=device, dtype=unet_dtype)
    t = torch.tensor([timestep], device=device)
    cond_clip, cond_mask = encode_clip_prompts([prompt])
    uncond_clip, uncond_mask = encode_clip_prompts([""])
    clip_h = torch.cat([cond_clip, uncond_clip], dim=0)
    clip_mask = torch.cat([cond_mask, uncond_mask], dim=0)
    noisy = torch.cat([latent, latent], dim=0)
    tt = torch.cat([t, t], dim=0)
    set_dual_attention_context(unet, gemma_encoder_hidden_states=None, gemma_attention_mask=None, clip_scale=1.0, gemma_scale=0.0)
    pred = unet(noisy, tt, encoder_hidden_states=clip_h, encoder_attention_mask=clip_mask).sample.float()
    cond_pred, uncond_pred = pred.chunk(2)
    delta = cond_pred - uncond_pred
    delta_norm = delta.pow(2).mean().sqrt().item()
    pred_norm = cond_pred.pow(2).mean().sqrt().item() + 1e-8
    rel = delta_norm / pred_norm
    print(f"Teacher text-delta norm={delta_norm:.6f}, relative={rel:.6f} for prompt: {prompt}")
    wandb.log({"diagnostics/teacher_text_delta_norm": delta_norm,
               "diagnostics/teacher_text_delta_relative": rel})
    return rel

@torch.no_grad()
def save_training_validation_grid(tag, step, clip_scale=0.0, gemma_scale=1.0, prompts=None, steps=None, guidance=None, seed=None):
    """Save a fixed validation grid during training using the same dual graph path."""
    from diffusers import DPMSolverMultistepScheduler
    import matplotlib.pyplot as plt
    from PIL import Image
    import numpy as np

    prompts = list(prompts or VAL_PROMPTS)
    steps = int(steps or VALIDATION_TRAINING_STEPS)
    guidance = float(guidance or VAL_GUIDANCE)
    seed = int(seed or VAL_SEED)
    was_training = unet.training
    unet.eval(); vae.eval(); gemma_model.eval(); clip_model.eval()
    infer_scheduler = DPMSolverMultistepScheduler.from_config(scheduler.config)
    vae_dtype = next(vae.parameters()).dtype

    def _decode(latents):
        latents = (latents / vae.config.scaling_factor).to(dtype=vae_dtype)
        img = vae.decode(latents).sample
        img = (img / 2 + 0.5).clamp(0, 1)
        img = img.cpu().permute(0, 2, 3, 1).float().numpy()
        return Image.fromarray((img[0] * 255).astype(np.uint8))

    images = []
    for prompt_idx, prompt in enumerate(prompts):
        gen = torch.Generator(device=device).manual_seed(seed + prompt_idx)
        infer_scheduler.set_timesteps(steps, device=device)
        cond_clip, cond_clip_mask = encode_clip_prompts([prompt])
        uncond_clip, uncond_clip_mask = encode_clip_prompts([""])
        cond_gemma, cond_gemma_mask = encode_gemma_prompts([prompt])
        uncond_gemma, uncond_gemma_mask = encode_gemma_prompts([""])
        clip_h = torch.cat([uncond_clip, cond_clip], dim=0)
        clip_mask = torch.cat([uncond_clip_mask, cond_clip_mask], dim=0)
        gemma_h = torch.cat([uncond_gemma, cond_gemma], dim=0)
        gemma_mask = torch.cat([uncond_gemma_mask, cond_gemma_mask], dim=0)
        latents = torch.randn(1, 4, 64, 64, generator=gen, device=device, dtype=unet_dtype) * infer_scheduler.init_noise_sigma
        set_dual_attention_context(unet, gemma_encoder_hidden_states=gemma_h, gemma_attention_mask=gemma_mask, clip_scale=clip_scale, gemma_scale=gemma_scale)
        for t in infer_scheduler.timesteps:
            inp = torch.cat([latents] * 2, dim=0)
            inp = infer_scheduler.scale_model_input(inp, t)
            pred = unet(inp, t, encoder_hidden_states=clip_h, encoder_attention_mask=clip_mask).sample
            u, c = pred.chunk(2)
            pred = u + guidance * (c - u)
            latents = infer_scheduler.step(pred, t, latents).prev_sample
        images.append(_decode(latents))

    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 5))
    if len(images) == 1:
        axes = [axes]
    for ax, img, ptxt in zip(axes, images, prompts):
        ax.imshow(img)
        ax.set_title(ptxt[:40] + "...", fontsize=10)
        ax.axis("off")
    fig.suptitle(f"{tag} step {step} c={clip_scale} g={gemma_scale}")
    plt.tight_layout()
    out_path = f"{DRIVE_OUT}/validation_{tag}_step_{int(step):06d}.png"
    plt.savefig(out_path, dpi=100)
    plt.show()
    wandb.log({f"validation/{tag}_step": int(step)})
    print(f"Saved training validation grid: {out_path}")
    if was_training:
        unet.train()
    return out_path

@torch.no_grad()
def test_dual_forward(prompt="a cat on a table"):
    clip_h, clip_mask = encode_clip_prompts(prompt)
    gemma_h, gemma_mask = encode_gemma_prompts(prompt)
    print(f"CLIP stats:  mean={clip_h.float().mean().item():.4f}, std={clip_h.float().std().item():.4f}, max={clip_h.float().abs().max().item():.4f}")
    print(f"Gemma stats: mean={gemma_h.float().mean().item():.4f}, std={gemma_h.float().std().item():.4f}, max={gemma_h.float().abs().max().item():.4f}")
    latents = torch.randn(1, 4, 64, 64, device=device, dtype=unet_dtype)
    t = torch.tensor([500], device=device)
    set_dual_attention_context(unet, gemma_encoder_hidden_states=None, gemma_attention_mask=None, clip_scale=1.0, gemma_scale=0.0)
    clip_only = unet(latents, t, encoder_hidden_states=clip_h, encoder_attention_mask=clip_mask).sample
    set_dual_attention_context(unet, gemma_encoder_hidden_states=gemma_h, gemma_attention_mask=gemma_mask, clip_scale=1.0, gemma_scale=0.0)
    dual_zero = unet(latents, t, encoder_hidden_states=clip_h, encoder_attention_mask=clip_mask).sample
    max_delta = (clip_only - dual_zero).float().abs().max().item()
    print(f"CLIP-only equivalence max delta: {max_delta:.8f}")
    assert max_delta < 1e-5, "Gemma scale 0 should preserve original CLIP path"
    set_dual_attention_context(unet, gemma_encoder_hidden_states=gemma_h, gemma_attention_mask=gemma_mask, clip_scale=0.0, gemma_scale=1.0)
    gemma_only = unet(latents, t, encoder_hidden_states=clip_h, encoder_attention_mask=clip_mask).sample
    assert torch.isfinite(gemma_only).all(), "Gemma-only forward produced NaN/Inf"
    print(f"Gemma-only output stats: mean={gemma_only.float().mean().item():.4f}, std={gemma_only.float().std().item():.4f}, max={gemma_only.float().abs().max().item():.4f}")
    set_dual_attention_context(unet, gemma_encoder_hidden_states=None, gemma_attention_mask=None, clip_scale=1.0, gemma_scale=0.0)
    return clip_only, gemma_only

test_dual_forward()
print("✓ Dual forward smoke test OK")

if RUN_DIAGNOSTICS:
    print("\n=== Mandatory pre-training diagnostics ===")
    clip_sens = compute_dual_prompt_sensitivity(DIAGNOSTIC_PROMPTS, clip_scale=1.0, gemma_scale=0.0, label="clip_only_teacher")
    gemma_untrained_sens = compute_dual_prompt_sensitivity(DIAGNOSTIC_PROMPTS, clip_scale=0.0, gemma_scale=1.0, label="gemma_only_untrained")
    layer_stats = gemma_layer_discriminability(DIAGNOSTIC_PROMPTS)
    attn_stats = gemma_attention_score_diagnostic(DIAGNOSTIC_PROMPTS[0])
    teacher_delta_rel = teacher_text_delta_baseline(DIAGNOSTIC_PROMPTS[0])
    print(f"Gemma/CLIP sensitivity ratio before training: {gemma_untrained_sens / max(clip_sens, 1e-8):.4f}")


## Section 4: LoRA Training

Freeze VAE + all UNet except LoRA on attn2.to_k/attn2.to_v

## Section 4A: Streaming Dataset

Streams `jackyhate/text-to-image-2M` with simple custom aspect-ratio bucketing.
This is **not** sd-scripts `BucketManager`; it is a minimal custom loop for proof-of-life training.
With `batch_size=1`, each sample may use its own bucket resolution without padding.


In [ ]:
# @title 4.2 Streaming IterableDataset + Shuffled DataLoader Helper
import io
import torch
from PIL import Image
from torch.utils.data import IterableDataset, DataLoader
from torchvision import transforms
from datasets import load_dataset

BUCKETS = [
    (512, 512), (512, 768), (768, 512),
    (448, 704), (704, 448), (384, 640), (640, 384),
]

def get_bucket(w, h):
    """Nearest aspect-ratio bucket."""
    target = w / h
    best, best_dist = None, float("inf")
    for bw, bh in BUCKETS:
        dist = abs(bw / bh - target)
        if dist < best_dist:
            best_dist, best = dist, (bw, bh)
    return best

class StreamingSDDataset(IterableDataset):
    """Stream images/captions from HF webdataset. Tokenization happens in encoder helpers."""
    def __init__(self, ds_iter, vae=None, max_samples=2000):
        self.ds_iter = ds_iter
        self.max_samples = int(max_samples)
        self.vae = vae

    def __iter__(self):
        import json as _json
        def _get_caption(sample):
            meta = sample.get("json", {})
            if isinstance(meta, bytes):
                meta = meta.decode("utf-8", errors="ignore")
            if isinstance(meta, str):
                try:
                    meta = _json.loads(meta)
                except Exception:
                    return ""
            if isinstance(meta, dict):
                return meta.get("prompt") or meta.get("caption") or meta.get("text") or ""
            return ""

        def _get_image(sample):
            for key in ["jpg", "jpeg", "png", "webp", "image"]:
                img = sample.get(key)
                if img is not None:
                    return img
            return None

        worker_info = torch.utils.data.get_worker_info()
        per_worker = self.max_samples
        if worker_info is not None:
            per_worker = max(1, self.max_samples // worker_info.num_workers)

        count = 0
        for sample in self.ds_iter:
            if count >= per_worker:
                break
            caption = _get_caption(sample)
            if not caption:
                continue
            img = _get_image(sample)
            if img is None:
                continue
            if isinstance(img, bytes):
                img = Image.open(io.BytesIO(img))
            img = img.convert("RGB")
            bw, bh = get_bucket(img.width, img.height)
            b_ratio = bw / bh
            w, h = img.size
            if w / h > b_ratio:
                new_w = int(h * b_ratio)
                img = img.crop(((w - new_w) // 2, 0, (w + new_w) // 2, h))
            else:
                new_h = int(w / b_ratio)
                img = img.crop((0, (h - new_h) // 2, w, (h + new_h) // 2))
            img = img.resize((bw, bh), Image.LANCZOS)
            img_tensor = transforms.ToTensor()(img) * 2 - 1
            yield {"image": img_tensor, "caption": caption}
            count += 1

MAX_SAMPLES = MAX_SAMPLES_WARMUP
STREAM_REPO = "jackyhate/text-to-image-2M"
print(f"Streaming from: {STREAM_REPO}")

def make_streaming_dataloader(phase, epoch, max_samples, batch_size=1):
    ds_full = load_dataset(STREAM_REPO, split="train", streaming=True)
    shuffle_seed = BASE_SEED + 1000 * int(phase) + int(epoch)
    if SHUFFLE_STREAMING:
        ds_full = ds_full.shuffle(buffer_size=SHUFFLE_BUFFER, seed=shuffle_seed)
    ds = StreamingSDDataset(ds_full, vae=vae, max_samples=max_samples)
    dl = DataLoader(ds, batch_size=batch_size, num_workers=0)
    print(f"DataLoader phase={phase} epoch={epoch} max_samples={max_samples} shuffle={SHUFFLE_STREAMING} seed={shuffle_seed}")
    wandb.log({f"data/phase_{phase}_epoch_{epoch}_shuffle_seed": shuffle_seed,
               f"data/phase_{phase}_epoch_{epoch}_max_samples": max_samples})
    return dl

# Lightweight preview. The training loops create fresh shuffled streams per epoch.
preview_stream = load_dataset(STREAM_REPO, streaming=True, split="train")
preview_sample = next(iter(preview_stream))
json_data = preview_sample.get("json", {})
prompt = json_data.get("prompt", "") if isinstance(json_data, dict) else str(json_data)
print(f"  Sample keys: {list(preview_sample.keys())}")
print(f"  Prompt: {prompt[:80]}")
dl = make_streaming_dataloader(phase=0, epoch=0, max_samples=min(MAX_SAMPLES_WARMUP, 8), batch_size=1)
print(f"Dataset helper ready: warmup={MAX_SAMPLES_WARMUP}, phaseB={MAX_SAMPLES_PHASEB}, {len(BUCKETS)} buckets")


## Section 4: Phase A — Full-Rank Gemma Branch Warmup
Train native Gemma `gemma_norm + to_k + to_v` full-rank before applying LoRA. Student forward is Gemma-only; CLIP is frozen teacher target. Text-delta loss makes the objective focus on CLIP's conditional-vs-unconditional contribution, not just generic denoising prior.


In [ ]:
# @title 4.0 Phase A: Full-Rank Gemma Branch Warmup + Text-Delta Distillation
from tqdm import tqdm

if not RUN_TRAINING or FULLRANK_EPOCHS <= 0:
    print(f"Skipping Phase A full-rank warmup because RUN_MODE={RUN_MODE}")
else:
    WARMUP_CLIP_SCALE = 0.0
    WARMUP_GEMMA_SCALE = 1.0

    trainable_params = freeze_all_but_gemma_branch(unet)
    print_trainable_summary(unet, label="phase_a_fullrank_start")
    optimizer = torch.optim.AdamW(trainable_params, lr=FULLRANK_LR, eps=1e-6)
    unet.train(); vae.eval(); gemma_model.eval(); clip_model.eval()

    print(f"Phase A full-rank Gemma warmup: {FULLRANK_EPOCHS} epochs")
    print(f"Dataset batch size: {TRAIN_BATCH_SIZE}; paired text-delta UNet batch size: {2 * TRAIN_BATCH_SIZE}")
    print(f"Phase A optimizer-step limit: {PHASE_A_MAX_OPT_STEPS}")
    phase_a_opt_step = 0
    if RUN_FIXED_VALIDATION_GRIDS and VALIDATION_EVERY_OPT_STEPS:
        save_training_validation_grid("phase_a_start_gemma_only", phase_a_opt_step, clip_scale=0.0, gemma_scale=1.0)
    stop_phase_a = False
    for epoch in range(FULLRANK_EPOCHS):
        dl = make_streaming_dataloader(phase=1, epoch=epoch, max_samples=MAX_SAMPLES_WARMUP, batch_size=TRAIN_BATCH_SIZE)
        total_loss = 0.0
        seen = 0
        progress = tqdm(dl, desc=f"PhaseA {epoch+1}/{FULLRANK_EPOCHS}")

        for batch in progress:
            captions = _as_prompt_list(batch["caption"])
            empty_captions = [""] * len(captions)
            img = batch["image"].to(device, dtype=unet_dtype)
            with torch.no_grad():
                latent = vae.encode(img).latent_dist.sample() * vae.config.scaling_factor
                assert torch.isfinite(latent).all(), "VAE latent has NaN/Inf"
                cond_clip_h, cond_clip_mask = encode_clip_prompts(captions)
                uncond_clip_h, uncond_clip_mask = encode_clip_prompts(empty_captions)
                cond_gemma_h, cond_gemma_mask = encode_gemma_prompts(captions)
                uncond_gemma_h, uncond_gemma_mask = encode_gemma_prompts(empty_captions)

            noise = torch.randn_like(latent)
            t = torch.randint(0, scheduler.config.num_train_timesteps, (latent.shape[0],), device=device).long()
            noisy = scheduler.add_noise(latent, noise, t)
            noisy_pair = torch.cat([noisy, noisy], dim=0)
            t_pair = torch.cat([t, t], dim=0)
            clip_h_pair = torch.cat([cond_clip_h, uncond_clip_h], dim=0)
            clip_mask_pair = torch.cat([cond_clip_mask, uncond_clip_mask], dim=0)
            gemma_h_pair = torch.cat([cond_gemma_h, uncond_gemma_h], dim=0)
            gemma_mask_pair = torch.cat([cond_gemma_mask, uncond_gemma_mask], dim=0)

            # Teacher: CLIP cond/uncond in one no-grad forward.
            with torch.no_grad():
                set_dual_attention_context(unet, gemma_encoder_hidden_states=None, gemma_attention_mask=None, clip_scale=1.0, gemma_scale=0.0)
                teacher_pair = unet(noisy_pair, t_pair, encoder_hidden_states=clip_h_pair, encoder_attention_mask=clip_mask_pair).sample.detach()
                teacher_cond, teacher_uncond = teacher_pair.chunk(2)
                teacher_delta = teacher_cond - teacher_uncond

            # Student: Gemma cond/uncond in one differentiable forward; CLIP branch skipped inside DualNativeAttention.
            set_dual_attention_context(unet, gemma_encoder_hidden_states=gemma_h_pair, gemma_attention_mask=gemma_mask_pair, clip_scale=WARMUP_CLIP_SCALE, gemma_scale=WARMUP_GEMMA_SCALE)
            student_pair = unet(noisy_pair, t_pair, encoder_hidden_states=clip_h_pair, encoder_attention_mask=clip_mask_pair).sample
            student_cond, student_uncond = student_pair.chunk(2)
            if not torch.isfinite(student_pair).all():
                raise RuntimeError("UNet prediction has NaN/Inf during Phase A")

            student_delta = student_cond - student_uncond
            loss_teacher = nn.functional.mse_loss(student_cond.float(), teacher_cond.float())
            loss_text_delta = nn.functional.mse_loss(student_delta.float(), teacher_delta.float()) if TEXT_DELTA_ENABLED else student_cond.new_tensor(0.0)
            loss_diffusion = nn.functional.mse_loss(student_cond.float(), noise.float())
            loss = loss_teacher + LAMBDA_TEXT_DELTA * loss_text_delta + LAMBDA_DIFFUSION * loss_diffusion
            if not torch.isfinite(loss):
                raise RuntimeError("Phase A loss is NaN/Inf")

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(trainable_params, 0.5)
            optimizer.step()
            phase_a_opt_step += 1
            if RUN_FIXED_VALIDATION_GRIDS and VALIDATION_EVERY_OPT_STEPS and phase_a_opt_step % VALIDATION_EVERY_OPT_STEPS == 0:
                save_training_validation_grid("phase_a_gemma_only", phase_a_opt_step, clip_scale=0.0, gemma_scale=1.0)
            if PHASE_A_MAX_OPT_STEPS is not None and phase_a_opt_step >= PHASE_A_MAX_OPT_STEPS:
                stop_phase_a = True

            total_loss += loss.item()
            seen += 1
            progress.set_postfix({"loss": f"{loss.item():.4f}", "delta": f"{loss_text_delta.item():.4f}"})
            if seen % 25 == 0:
                wandb.log({
                    "phase_a/loss": loss.item(),
                    "phase_a/loss_teacher": loss_teacher.item(),
                    "phase_a/loss_text_delta": loss_text_delta.item(),
                    "phase_a/loss_diffusion": loss_diffusion.item(),
                    "phase_a/clip_scale": WARMUP_CLIP_SCALE,
                    "phase_a/gemma_scale": WARMUP_GEMMA_SCALE,
                    "phase_a/step": seen + epoch * MAX_SAMPLES_WARMUP,
                    "phase_a/optimizer_step": phase_a_opt_step,
                })
            if stop_phase_a:
                print(f"Stopping Phase A at configured optimizer-step limit: {phase_a_opt_step}")
                break

        avg = total_loss / max(seen, 1)
        print(f"Phase A epoch {epoch+1}: avg_loss = {avg:.4f}")
        wandb.log({"phase_a/epoch_loss": avg, "phase_a/epoch": epoch + 1, "phase_a/optimizer_step": phase_a_opt_step})
        if stop_phase_a:
            break

    if RUN_FIXED_VALIDATION_GRIDS and VALIDATION_EVERY_OPT_STEPS:
        save_training_validation_grid("phase_a_end_gemma_only", phase_a_opt_step, clip_scale=0.0, gemma_scale=1.0)

    fullrank_path = f"{DRIVE_OUT}/gemma3_sd_phase_a_fullrank.pt"
    torch.save({
        "unet_state_dict": {k: v.detach().cpu() for k, v in unet.state_dict().items()},
        "unet_config": dict(unet.config),
        "architecture": "DualNativeAttention phase A full-rank Gemma K/V",
        "gemma_hidden_size": gemma_hidden_size,
        "max_gemma_len": MAX_GEMMA_LEN,
        "run_config": RUN_CONFIG,
    }, fullrank_path)
    print(f"Phase A full-rank checkpoint saved: {fullrank_path}")

set_dual_attention_context(unet, gemma_encoder_hidden_states=None, gemma_attention_mask=None, clip_scale=0.0, gemma_scale=1.0)
print("Phase A complete. Next: optional LoRA/to_out continuation.")


In [ ]:
# @title 4.1 Manual LoRA for Dual Native Attention Branches
class ManualLoRA(nn.Module):
    """LoRA wrapper for a single Linear layer."""
    def __init__(self, base_linear, rank=8, alpha=16):
        super().__init__()
        self.base = base_linear
        self.rank = rank
        self.scaling = alpha / rank
        in_f, out_f = base_linear.in_features, base_linear.out_features
        self.lora_A = nn.Linear(in_f, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_f, bias=False)
        nn.init.normal_(self.lora_A.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.lora_B.weight)
        self.lora_A = self.lora_A.to(dtype=base_linear.weight.dtype, device=base_linear.weight.device)
        self.lora_B = self.lora_B.to(dtype=base_linear.weight.dtype, device=base_linear.weight.device)
        for p in base_linear.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.base(x) + self.lora_B(self.lora_A(x)) * self.scaling

LORA_RANK = 8
LORA_ALPHA = 16

if not RUN_TRAINING or PHASEB_EPOCHS <= 0:
    print(f"Skipping LoRA wrapping because RUN_MODE={RUN_MODE}")
else:
    # Freeze all, then wrap Gemma branch K/V only. CLIP branch remains frozen teacher/scaffold.
    for p in unet.parameters():
        p.requires_grad = False

    lora_count = 0
    for module in unet.modules():
        if is_dual_native_attention_module(module):
            if not isinstance(module.gemma_attn.to_k, ManualLoRA):
                module.gemma_attn.to_k = ManualLoRA(module.gemma_attn.to_k, rank=LORA_RANK, alpha=LORA_ALPHA)
                module.gemma_attn.to_v = ManualLoRA(module.gemma_attn.to_v, rank=LORA_RANK, alpha=LORA_ALPHA)
                lora_count += 2
            for p in module.gemma_norm.parameters():
                p.requires_grad = True
            # Strict LoRA scope at wrapping point: LoRA A/B only plus gemma_norm.
            for n, p in module.gemma_attn.named_parameters():
                if ("to_k.lora_A" in n or "to_k.lora_B" in n or "to_v.lora_A" in n or "to_v.lora_B" in n):
                    p.requires_grad = True

    print(f"Gemma LoRA layers (K/V only): {lora_count}")
    print_trainable_summary(unet, label="phase_b_after_lora_wrap_before_to_out")
    wandb.config.update({"lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA, "lora_layers": lora_count}, allow_val_change=True)


## Section 4B: Phase B — K/V LoRA + Gemma Output Projection
After Phase A establishes a full-rank Gemma K/V path, wrap K/V with LoRA and unfreeze `gemma_attn.to_out[0]`. This is not pure 0.04% LoRA; the trainable count is printed again after `to_out` is enabled.


In [ ]:
# @title 4.4 Phase B: Gemma-Only Continuation + Text-Delta Teacher Decay
from tqdm import tqdm

if not RUN_TRAINING or PHASEB_EPOCHS <= 0:
    print(f"Skipping Phase B because RUN_MODE={RUN_MODE}")
else:
    # Continue Gemma-only training before any CLIP/Gemma mixing.
    # Phase B scope: K/V LoRA + gemma_norm + Gemma branch to_out.
    unfreeze_gemma_to_out(unet)
    trainable_params = [p for p in unet.parameters() if p.requires_grad]
    print_trainable_summary(unet, label="phase_b_after_unfreeze_to_out")
    optimizer = torch.optim.AdamW(trainable_params, lr=PHASEB_LR, weight_decay=0.01, eps=1e-6)
    optimizer.zero_grad(set_to_none=True)
    for p in unet.parameters():
        p.grad = None

    unet.train(); vae.eval(); gemma_model.eval(); clip_model.eval()
    PHASEB_CLIP_SCALE = 0.0
    PHASEB_GEMMA_SCALE = 1.0

    TEACHER_START = 1.0
    TEACHER_END = 0.1
    global_step = 0
    optimizer_step = 0

    print(f"Phase B Gemma-only training {PHASEB_EPOCHS} epochs, streaming from {STREAM_REPO}")
    print(f"Dataset batch size: {TRAIN_BATCH_SIZE}; paired text-delta UNet batch size: {2 * TRAIN_BATCH_SIZE}")
    print(f"Grad accum: {GRADIENT_ACCUMULATION_STEPS}x")
    print(f"Phase B optimizer-step limit: {PHASE_B_MAX_OPT_STEPS}")
    print(f"Teacher loss decay: start={TEACHER_START:.3f} -> end={TEACHER_END:.3f} over {TEACHER_DECAY_STEPS} optimizer steps")
    print("Student forward stays Gemma-only (clip_scale=0.0, gemma_scale=1.0); text-delta loss targets CLIP conditional-unconditional delta.")

    def teacher_lambda_at_step(opt_step: int) -> float:
        frac = min(max(opt_step / max(TEACHER_DECAY_STEPS, 1), 0.0), 1.0)
        return TEACHER_START + frac * (TEACHER_END - TEACHER_START)

    if RUN_FIXED_VALIDATION_GRIDS and VALIDATION_EVERY_OPT_STEPS:
        save_training_validation_grid("phase_b_start_gemma_only", optimizer_step, clip_scale=0.0, gemma_scale=1.0)
    stop_phase_b = False

    try:
        for epoch in range(PHASEB_EPOCHS):
            dl = make_streaming_dataloader(phase=2, epoch=epoch, max_samples=MAX_SAMPLES_PHASEB, batch_size=TRAIN_BATCH_SIZE)
            epoch_loss = 0.0
            samples_seen = 0
            progress = tqdm(dl, desc=f"PhaseB {epoch+1}/{PHASEB_EPOCHS}")

            for batch in progress:
                captions = _as_prompt_list(batch["caption"])
                empty_captions = [""] * len(captions)
                img = batch["image"].to(device, dtype=unet_dtype)
                with torch.no_grad():
                    latent = vae.encode(img).latent_dist.sample() * vae.config.scaling_factor
                    assert torch.isfinite(latent).all(), "VAE latent has NaN/Inf"
                    cond_clip_h, cond_clip_mask = encode_clip_prompts(captions)
                    uncond_clip_h, uncond_clip_mask = encode_clip_prompts(empty_captions)
                    cond_gemma_h, cond_gemma_mask = encode_gemma_prompts(captions)
                    uncond_gemma_h, uncond_gemma_mask = encode_gemma_prompts(empty_captions)

                noise = torch.randn_like(latent)
                t = torch.randint(0, scheduler.config.num_train_timesteps, (latent.shape[0],), device=device).long()
                noisy = scheduler.add_noise(latent, noise, t)
                noisy_pair = torch.cat([noisy, noisy], dim=0)
                t_pair = torch.cat([t, t], dim=0)
                clip_h_pair = torch.cat([cond_clip_h, uncond_clip_h], dim=0)
                clip_mask_pair = torch.cat([cond_clip_mask, uncond_clip_mask], dim=0)
                gemma_h_pair = torch.cat([cond_gemma_h, uncond_gemma_h], dim=0)
                gemma_mask_pair = torch.cat([cond_gemma_mask, uncond_gemma_mask], dim=0)

                with torch.no_grad():
                    set_dual_attention_context(unet, gemma_encoder_hidden_states=None, gemma_attention_mask=None, clip_scale=1.0, gemma_scale=0.0)
                    teacher_pair = unet(noisy_pair, t_pair, encoder_hidden_states=clip_h_pair, encoder_attention_mask=clip_mask_pair).sample.detach()
                    teacher_cond, teacher_uncond = teacher_pair.chunk(2)
                    teacher_delta = teacher_cond - teacher_uncond

                set_dual_attention_context(unet, gemma_encoder_hidden_states=gemma_h_pair, gemma_attention_mask=gemma_mask_pair, clip_scale=PHASEB_CLIP_SCALE, gemma_scale=PHASEB_GEMMA_SCALE)
                student_pair = unet(noisy_pair, t_pair, encoder_hidden_states=clip_h_pair, encoder_attention_mask=clip_mask_pair).sample
                student_cond, student_uncond = student_pair.chunk(2)
                if not torch.isfinite(student_pair).all():
                    raise RuntimeError("UNet prediction has NaN/Inf during Phase B")

                student_delta = student_cond - student_uncond
                lambda_teacher = teacher_lambda_at_step(optimizer_step)
                loss_teacher_raw = nn.functional.mse_loss(student_cond.float(), teacher_cond.float())
                loss_text_delta_raw = nn.functional.mse_loss(student_delta.float(), teacher_delta.float()) if TEXT_DELTA_ENABLED else student_cond.new_tensor(0.0)
                loss_diffusion_raw = nn.functional.mse_loss(student_cond.float(), noise.float())
                loss_total_raw = lambda_teacher * loss_teacher_raw + LAMBDA_TEXT_DELTA * loss_text_delta_raw + LAMBDA_DIFFUSION * loss_diffusion_raw
                loss = loss_total_raw / GRADIENT_ACCUMULATION_STEPS
                if not torch.isfinite(loss):
                    raise RuntimeError("Phase B loss is NaN/Inf")

                loss.backward()
                if (global_step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    nn.utils.clip_grad_norm_(trainable_params, 0.5)
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    optimizer_step += 1
                    if RUN_FIXED_VALIDATION_GRIDS and VALIDATION_EVERY_OPT_STEPS and optimizer_step % VALIDATION_EVERY_OPT_STEPS == 0:
                        save_training_validation_grid("phase_b_gemma_only", optimizer_step, clip_scale=0.0, gemma_scale=1.0)
                    if PHASE_B_MAX_OPT_STEPS is not None and optimizer_step >= PHASE_B_MAX_OPT_STEPS:
                        stop_phase_b = True

                shown_loss = loss_total_raw.item()
                epoch_loss += shown_loss
                samples_seen += 1
                progress.set_postfix({"loss": f"{shown_loss:.4f}", "delta": f"{loss_text_delta_raw.item():.4f}", "opt_step": optimizer_step})

                if global_step % 20 == 0:
                    wandb.log({
                        "phase_b/loss_total_raw": shown_loss,
                        "phase_b/loss_scaled": loss.item(),
                        "phase_b/lambda_teacher": float(lambda_teacher),
                        "phase_b/lambda_diffusion": float(LAMBDA_DIFFUSION),
                        "phase_b/lambda_text_delta": float(LAMBDA_TEXT_DELTA),
                        "phase_b/loss_teacher_raw": loss_teacher_raw.item(),
                        "phase_b/loss_text_delta_raw": loss_text_delta_raw.item(),
                        "phase_b/loss_diffusion_raw": loss_diffusion_raw.item(),
                        "phase_b/step": global_step,
                        "phase_b/optimizer_step": optimizer_step,
                        "phase_b/clip_scale": PHASEB_CLIP_SCALE,
                        "phase_b/gemma_scale": PHASEB_GEMMA_SCALE,
                    })
                global_step += 1
                if stop_phase_b:
                    print(f"Stopping Phase B at configured optimizer-step limit: {optimizer_step}")
                    break

            if samples_seen % GRADIENT_ACCUMULATION_STEPS != 0 and not stop_phase_b:
                nn.utils.clip_grad_norm_(trainable_params, 0.5)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                optimizer_step += 1
                if RUN_FIXED_VALIDATION_GRIDS and VALIDATION_EVERY_OPT_STEPS and optimizer_step % VALIDATION_EVERY_OPT_STEPS == 0:
                    save_training_validation_grid("phase_b_gemma_only", optimizer_step, clip_scale=0.0, gemma_scale=1.0)
                if PHASE_B_MAX_OPT_STEPS is not None and optimizer_step >= PHASE_B_MAX_OPT_STEPS:
                    stop_phase_b = True

            epoch_avg = epoch_loss / max(samples_seen, 1)
            print(f"Phase B epoch {epoch+1}: avg_loss = {epoch_avg:.4f}")
            wandb.log({"phase_b/epoch": epoch + 1, "phase_b/epoch_loss": epoch_avg, "phase_b/epoch_optimizer_steps": optimizer_step})
            if stop_phase_b:
                break

    except KeyboardInterrupt:
        print("Phase B interrupted. Progress retained in current model state.")

if RUN_TRAINING and RUN_FIXED_VALIDATION_GRIDS and VALIDATION_EVERY_OPT_STEPS:
    save_training_validation_grid("phase_b_end_gemma_only", optimizer_step if 'optimizer_step' in globals() else 0, clip_scale=0.0, gemma_scale=1.0)
set_dual_attention_context(unet, gemma_encoder_hidden_states=None, gemma_attention_mask=None, clip_scale=0.0, gemma_scale=1.0)
print("Phase B complete. Next: validation grids, checkpoints, and reloaded-pruned final proof.")


## Section 5: Inference

Generate images with the Gemma-conditioned SD.

In [ ]:
# @title 5.1 Fixed Validation Grids + Prompt Sensitivity Diagnostics
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from diffusers import DPMSolverMultistepScheduler
from tqdm import tqdm

vae.to(device).eval(); unet.to(device).eval(); gemma_model.eval(); clip_model.eval()
infer_scheduler = DPMSolverMultistepScheduler.from_config(scheduler.config)
vae_dtype = next(vae.parameters()).dtype

@torch.no_grad()
def make_dual_condition(prompts, include_clip=True, include_gemma=True):
    clip_h, clip_mask = encode_clip_prompts(prompts)
    gemma_h, gemma_mask = encode_gemma_prompts(prompts)
    if not include_clip:
        clip_h = torch.zeros_like(clip_h)
    if not include_gemma:
        gemma_h = torch.zeros_like(gemma_h)
    return clip_h, clip_mask, gemma_h, gemma_mask

@torch.no_grad()
def decode_latents_to_image(latents):
    latents = (latents / vae.config.scaling_factor).to(dtype=vae_dtype)
    img = vae.decode(latents).sample
    img = (img / 2 + 0.5).clamp(0, 1)
    img = img.cpu().permute(0, 2, 3, 1).float().numpy()
    return Image.fromarray((img[0] * 255).astype(np.uint8))

@torch.no_grad()
def generate_dual(prompt, steps=VAL_STEPS, guidance=VAL_GUIDANCE, seed=VAL_SEED, clip_scale=0.0, gemma_scale=1.0):
    gen = torch.Generator(device=device).manual_seed(seed)
    infer_scheduler.set_timesteps(steps, device=device)
    cond_clip, cond_clip_mask, cond_gemma, cond_gemma_mask = make_dual_condition([prompt])
    uncond_clip, uncond_clip_mask, uncond_gemma, uncond_gemma_mask = make_dual_condition([""])
    clip_h = torch.cat([uncond_clip, cond_clip], dim=0)
    clip_mask = torch.cat([uncond_clip_mask, cond_clip_mask], dim=0)
    gemma_h = torch.cat([uncond_gemma, cond_gemma], dim=0)
    gemma_mask = torch.cat([uncond_gemma_mask, cond_gemma_mask], dim=0)
    latents = torch.randn(1, 4, 64, 64, generator=gen, device=device, dtype=unet_dtype) * infer_scheduler.init_noise_sigma
    set_dual_attention_context(unet, gemma_encoder_hidden_states=gemma_h, gemma_attention_mask=gemma_mask, clip_scale=clip_scale, gemma_scale=gemma_scale)
    for t in tqdm(infer_scheduler.timesteps, desc=f"Generating dual c={clip_scale} g={gemma_scale}"):
        inp = torch.cat([latents] * 2, dim=0)
        inp = infer_scheduler.scale_model_input(inp, t)
        pred = unet(inp, t, encoder_hidden_states=clip_h, encoder_attention_mask=clip_mask).sample
        u, p = pred.chunk(2)
        pred = u + guidance * (p - u)
        latents = infer_scheduler.step(pred, t, latents).prev_sample
    return decode_latents_to_image(latents)

@torch.no_grad()
def generate_gemma_only_from_unet(target_unet, prompt, steps=VAL_STEPS, guidance=VAL_GUIDANCE, seed=VAL_SEED, desc="Gemma-only"):
    gen = torch.Generator(device=device).manual_seed(seed)
    infer_scheduler.set_timesteps(steps, device=device)
    cond_gemma, cond_mask = encode_gemma_prompts([prompt])
    uncond_gemma, uncond_mask = encode_gemma_prompts([""])
    gemma_h = torch.cat([uncond_gemma, cond_gemma], dim=0)
    gemma_mask = torch.cat([uncond_mask, cond_mask], dim=0)
    latents = torch.randn(1, 4, 64, 64, generator=gen, device=device, dtype=unet_dtype) * infer_scheduler.init_noise_sigma
    for t in tqdm(infer_scheduler.timesteps, desc=f"Generating {desc}"):
        inp = torch.cat([latents] * 2, dim=0)
        inp = infer_scheduler.scale_model_input(inp, t)
        pred = target_unet(inp, t, encoder_hidden_states=gemma_h, encoder_attention_mask=gemma_mask).sample
        u, p = pred.chunk(2)
        pred = u + guidance * (p - u)
        latents = infer_scheduler.step(pred, t, latents).prev_sample
    return decode_latents_to_image(latents)

@torch.no_grad()
def prompt_sensitivity(prompts, seed=123, timestep=500, clip_scale=0.0, gemma_scale=1.0, label="gemma_only"):
    return compute_dual_prompt_sensitivity(prompts, seed=seed, timestep=timestep, clip_scale=clip_scale, gemma_scale=gemma_scale, label=label)

@torch.no_grad()
def save_validation_grid(images, prompts, path, title=None):
    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 5))
    if len(images) == 1:
        axes = [axes]
    for ax, img, ptxt in zip(axes, images, prompts):
        ax.imshow(img)
        ax.set_title(ptxt[:40] + "...", fontsize=10)
        ax.axis("off")
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(path, dpi=100)
    plt.show()
    print(f"Saved validation grid: {path}")

print("Prompt sensitivity baselines on fixed validation prompts:")
clip_mean = prompt_sensitivity(VAL_PROMPTS, clip_scale=1.0, gemma_scale=0.0, label="clip_only_validation")
gemma_mean = prompt_sensitivity(VAL_PROMPTS, clip_scale=0.0, gemma_scale=1.0, label="gemma_only_validation")
print(f"Gemma/CLIP validation sensitivity ratio: {gemma_mean / max(clip_mean, 1e-8):.4f}")
wandb.log({"validation/gemma_clip_sensitivity_ratio": gemma_mean / max(clip_mean, 1e-8)})

if RUN_FIXED_VALIDATION_GRIDS:
    clip_imgs = []
    gemma_imgs = []
    for ptxt in VAL_PROMPTS:
        print(f"Generating CLIP-only baseline: {ptxt}")
        clip_imgs.append(generate_dual(ptxt, clip_scale=1.0, gemma_scale=0.0))
        print(f"Generating dual Gemma-only: {ptxt}")
        gemma_imgs.append(generate_dual(ptxt, clip_scale=0.0, gemma_scale=1.0))
    save_validation_grid(clip_imgs, VAL_PROMPTS, f"{DRIVE_OUT}/validation_clip_only_baseline.png", "CLIP-only baseline")
    save_validation_grid(gemma_imgs, VAL_PROMPTS, f"{DRIVE_OUT}/validation_dual_gemma_only.png", "Dual graph Gemma-only")


In [ ]:
# @title 5.2 Save Dual + Pruned Gemma-Only Checkpoints
import os

if not RUN_TRAINING:
    print(f"Skipping checkpoint save/prune because RUN_MODE={RUN_MODE}. Diagnostic mode should not write final artifacts from an untrained scaffold.")
else:
    # Merge LoRA wrappers first so both dual and pruned checkpoints strict-reload into plain Linear modules.
    def merge_lora_linear(m):
        if not isinstance(m, ManualLoRA):
            return m
        base = m.base
        delta = (m.lora_B.weight @ m.lora_A.weight) * m.scaling
        base.weight.data.add_(delta.to(device=base.weight.device, dtype=base.weight.dtype))
        return base

    def merge_all_lora(module):
        for child_name, child in list(module.named_children()):
            if isinstance(child, ManualLoRA):
                setattr(module, child_name, merge_lora_linear(child))
            else:
                merge_all_lora(child)

    merge_all_lora(unet)
    print("LoRA wrappers merged before checkpoint save; strict reload graphs use plain Linear modules.")

    # Dual checkpoint: debug/resume only; needs this notebook's DualNativeAttention class to reload.
    dual_path = f"{DRIVE_OUT}/gemma3_sd_dual_native_checkpoint.pt"
    torch.save({
        "unet_state_dict": {k: v.detach().cpu() for k, v in unet.state_dict().items()},
        "unet_config": dict(unet.config),
        "architecture": "DualNativeAttention(CLIP scaffold + native Gemma branch)",
        "artifact_role": "debug_resume_only",
        "gemma_model_id": gemma_path,
        "clip_model_id": CLIP_ID,
        "gemma_hidden_size": gemma_hidden_size,
        "clip_hidden_size": clip_hidden_size,
        "max_gemma_len": MAX_GEMMA_LEN,
        "run_config": RUN_CONFIG,
        "uses_kv_bias": True,
        "merged_lora": True,
        "reload_note": "Rebuild SD1.5 UNet, apply_dual_native_attention(unet, gemma_hidden_size), then load_state_dict(strict=True).",
    }, dual_path)
    print(f"Dual merged checkpoint saved: {dual_path}")

    # Pruned checkpoint: physically removes CLIP branch from live UNet while preserving Gemma LayerNorm.
    # This is intentionally a terminal notebook step: after this cell the live UNet is Gemma-only,
    # so rerun cells 3.x before re-running dual-graph diagnostics.
    pruned_count = prune_to_gemma_only(unet, gemma_hidden_size)
    pruned_path = f"{DRIVE_OUT}/gemma3_sd_gemma_only_pruned.pt"
    torch.save({
        "unet_state_dict": {k: v.detach().cpu() for k, v in unet.state_dict().items()},
        "unet_config": dict(unet.config),
        "architecture": "GemmaOnlyAttention wrapper; no CLIP branch in inference graph",
        "artifact_role": "final_inference",
        "gemma_model_id": gemma_path,
        "gemma_hidden_size": gemma_hidden_size,
        "new_cross_attention_dim": gemma_hidden_size,
        "max_gemma_len": MAX_GEMMA_LEN,
        "run_config": RUN_CONFIG,
        "uses_kv_bias": True,
        "merged_lora": True,
        "pruned_dual_modules": pruned_count,
        "reload_note": "Rebuild SD1.5 UNet, apply_dual_native_attention(unet, gemma_hidden_size), prune_to_gemma_only(unet), then load_state_dict(strict=True). No CLIP required for inference.",
    }, pruned_path)
    size_gb = os.path.getsize(pruned_path) / 1e9
    print(f"Gemma-only pruned checkpoint saved: {pruned_path} ({size_gb:.2f} GB)")
    print(f"Pruned modules: {pruned_count}")


In [ ]:
# @title 5.3 Final Proof from Freshly Reloaded Pruned Checkpoint
# Final artifact evidence must come from reloaded pruned UNet, not live dual graph.

if RUN_FINAL_PROOF and RUN_TRAINING:
    pruned_path = f"{DRIVE_OUT}/gemma3_sd_gemma_only_pruned.pt"
    pruned_ckpt = torch.load(pruned_path, map_location="cpu")

    # Rebuild from the same StyleJourney base checkpoint before applying Gemma surgery.
    reloaded_pruned_unet, _, _ = load_stylejourney_components(torch_dtype=unet_dtype, target_device=device)
    n = apply_dual_native_attention(reloaded_pruned_unet, pruned_ckpt["gemma_hidden_size"])
    pcount = prune_to_gemma_only(reloaded_pruned_unet, pruned_ckpt["gemma_hidden_size"])
    print(f"Reload pruned surgery applied: dual={n}, pruned={pcount}")

    reloaded_pruned_unet.load_state_dict(pruned_ckpt["unet_state_dict"], strict=True)
    reloaded_pruned_unet.eval()
    dual_left = count_modules_by_predicate(reloaded_pruned_unet, is_dual_native_attention_module)
    gemma_only_count = count_modules_by_predicate(reloaded_pruned_unet, is_gemma_only_attention_module)
    print(f"Reloaded module counts: DualNativeAttention={dual_left}, GemmaOnlyAttention={gemma_only_count}")
    assert dual_left == 0, "Pruned model should have zero DualNativeAttention modules"
    assert gemma_only_count == pcount, "Unexpected GemmaOnlyAttention count after reload"
    print("Pruned Gemma-only checkpoint reload: PASS (strict=True)")

    @torch.no_grad()
    def prompt_sensitivity_gemma_only_unet(target_unet, prompts, seed=123, timestep=500, label="reloaded_pruned"):
        gen = torch.Generator(device=device).manual_seed(seed)
        latent = torch.randn(1, 4, 64, 64, generator=gen, device=device, dtype=unet_dtype)
        t = torch.tensor([timestep], device=device)
        preds = []
        for ptxt in prompts:
            gh, gm = encode_gemma_prompts([ptxt])
            pred = target_unet(latent, t, encoder_hidden_states=gh, encoder_attention_mask=gm).sample.float()
            preds.append(pred)
        base = preds[0].pow(2).mean().sqrt().item() + 1e-8
        vals = []
        for i in range(len(prompts)):
            for j in range(i + 1, len(prompts)):
                diff = (preds[i] - preds[j]).pow(2).mean().sqrt().item() / base
                vals.append(diff)
                print(f"[{label}] {i} vs {j}: relative diff = {diff:.6f}")
        mean_val = float(np.mean(vals)) if vals else 0.0
        wandb.log({f"validation/{label}_mean_relative_diff": mean_val})
        return mean_val

    with torch.no_grad():
        test_latent = torch.randn(2, 4, 64, 64, device=device, dtype=unet_dtype)
        test_t = torch.tensor([500, 500], device=device).long()
        test_gemma_h, test_gemma_mask = encode_gemma_prompts(["a small red car", ""])
        test_pred = reloaded_pruned_unet(test_latent, test_t, encoder_hidden_states=test_gemma_h, encoder_attention_mask=test_gemma_mask).sample
        assert torch.isfinite(test_pred).all(), "Reloaded pruned forward produced NaN/Inf"
    print("Reloaded pruned finite forward: PASS")

    print("Prompt sensitivity (reloaded pruned Gemma-only):")
    reloaded_mean = prompt_sensitivity_gemma_only_unet(reloaded_pruned_unet, VAL_PROMPTS, label="reloaded_pruned")

    final_imgs = []
    for ptxt in VAL_PROMPTS:
        print(f"Generating reloaded-pruned Gemma-only: {ptxt}")
        final_imgs.append(generate_gemma_only_from_unet(reloaded_pruned_unet, ptxt, desc="reloaded-pruned Gemma-only"))
    final_grid = f"{DRIVE_OUT}/samples_reloaded_pruned_gemma_only.png"
    save_validation_grid(final_imgs, VAL_PROMPTS, final_grid, "Reloaded pruned Gemma-only")
    print(f"Final proof sample grid saved: {final_grid}")
else:
    print("Final proof skipped. Requires RUN_FINAL_PROOF=True and RUN_TRAINING=True so diagnostic-only runs do not reload untrained artifacts.")


## Current Contract / Handoff Notes

This notebook is one integrated diagnostic + training artifact for SD1.5 CLIP → Gemma conditioning replacement.

Non-negotiable contract:
- Student forward is Gemma-only: `clip_scale=0.0`, `gemma_scale=1.0`.
- CLIP is allowed only as a frozen teacher/scaffold during training diagnostics/losses.
- Do not use the failed CLIP→Gemma linear bake path.
- Gemma masks are propagated in training, generation, and pruned inference.
- Gemma Q and OUT are copied from the original CLIP branch; Gemma K/V are the native trainable text-side path.
- Text-delta loss trains the student to match CLIP's conditional-unconditional effect.
- Final proof must come from a freshly reloaded pruned Gemma-only checkpoint.

Quality gate:
- Treat lower MSE as insufficient evidence. Continue scaling only if Gemma/CLIP prompt-sensitivity ratio and fixed validation grids improve.
- If sensitivity stays flat while loss falls, stop and change representation/objective (for example Gemma layer choice), not whole-UNet finetune.

Artifact status:
- The pruned checkpoint is a custom Gemma-only UNet checkpoint requiring this notebook's surgery/reload helper.
- It is not yet a standalone CLIP-free Diffusers pipeline.

- Cell 5.2 prunes the live UNet as a terminal save step; rerun cells 3.x before re-running dual-graph diagnostics after saving.

- `TRAIN_BATCH_SIZE` default is 2. Text-delta uses cond/uncond pairs, so actual UNet batch is `2 * TRAIN_BATCH_SIZE`; increase to 3/4 only after watching peak VRAM.

- `short_train` is a controlled falsifier: Phase A stops at 100 optimizer steps, Phase B stops at 200 optimizer steps, and fixed Gemma-only grids are saved every 50 optimizer steps by default.
